# ReviewAgent — Stage 1 Classifier Results

This notebook visualizes:
1. Single-split evaluation results (80/20)
2. 5-fold stratified cross-validation results
3. Per-label performance analysis
4. Cross-fold stability

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

MODEL_DIR = Path("../models/stage1_classifier")
eval_results = json.loads((MODEL_DIR / "eval_results.json").read_text())
kfold_results = json.loads((MODEL_DIR / "kfold_results.json").read_text())

## 1. Single-Split vs 5-Fold CV Comparison

In [ ]:
metrics = ["F1 Micro", "F1 Macro", "Precision", "Recall"]
single = [
    eval_results["eval_metrics"]["eval_f1_micro"],
    eval_results["eval_metrics"]["eval_f1_macro"],
    eval_results["eval_metrics"]["eval_precision"],
    eval_results["eval_metrics"]["eval_recall"],
]
kfold_mean = [
    kfold_results["summary"]["f1_micro_mean"],
    kfold_results["summary"]["f1_macro_mean"],
    kfold_results["summary"]["precision_mean"],
    kfold_results["summary"]["recall_mean"],
]
kfold_std = [
    kfold_results["summary"]["f1_micro_std"],
    kfold_results["summary"]["f1_macro_std"],
    kfold_results["summary"]["precision_std"],
    kfold_results["summary"]["recall_std"],
]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, single, width, label="Single 80/20 Split", color="steelblue")
bars2 = ax.bar(x + width/2, kfold_mean, width, yerr=kfold_std, capsize=5,
               label="5-Fold CV (mean ± std)", color="coral")

ax.set_ylabel("Score")
ax.set_title("Single Split vs 5-Fold Stratified Cross-Validation")
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0.6, 0.85)
ax.legend()

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig("../notebooks/fig_single_vs_kfold.png", dpi=150, bbox_inches='tight')
plt.show()

## 2. Per-Label F1 Across Folds

In [ ]:
labels = list(kfold_results["per_label_average"].keys())
means = [kfold_results["per_label_average"][l]["f1_mean"] for l in labels]
stds = [kfold_results["per_label_average"][l]["f1_std"] for l in labels]
supports = [kfold_results["per_label_average"][l]["avg_support"] for l in labels]

# Sort by mean F1
order = np.argsort(means)
labels_s = [labels[i] for i in order]
means_s = [means[i] for i in order]
stds_s = [stds[i] for i in order]
supports_s = [supports[i] for i in order]

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#d62728' if m < 0.6 else '#ff7f0e' if m < 0.75 else '#2ca02c' for m in means_s]
bars = ax.barh(labels_s, means_s, xerr=stds_s, capsize=5, color=colors, edgecolor="white")
ax.set_xlabel("F1 Score (mean ± std across 5 folds)")
ax.set_title("Per-Label F1 Score — 5-Fold Stratified CV")
ax.set_xlim(0, 1.15)

for i, (m, s, sup) in enumerate(zip(means_s, stds_s, supports_s)):
    ax.text(m + s + 0.02, i, f'{m:.3f} ± {s:.3f}  (n={sup:.0f})', va='center', fontsize=9)

ax.axvline(0.5, color='gray', linestyle=':', alpha=0.5, label="F1 = 0.5")
ax.axvline(0.8, color='gray', linestyle='--', alpha=0.5, label="F1 = 0.8")
ax.legend(loc='lower right')

plt.tight_layout()
plt.savefig("../notebooks/fig_per_label_f1.png", dpi=150, bbox_inches='tight')
plt.show()

## 3. Cross-Fold Stability (F1 Macro per Fold)

In [ ]:
folds = kfold_results["per_fold"]
fold_nums = [f["fold"] for f in folds]
f1_micros = [f["f1_micro"] for f in folds]
f1_macros = [f["f1_macro"] for f in folds]
precisions = [f["precision"] for f in folds]
recalls = [f["recall"] for f in folds]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# F1 per fold
axes[0].plot(fold_nums, f1_micros, 'o-', label="F1 Micro", color="steelblue", markersize=8)
axes[0].plot(fold_nums, f1_macros, 's-', label="F1 Macro", color="coral", markersize=8)
axes[0].axhline(np.mean(f1_macros), color="coral", linestyle="--", alpha=0.5)
axes[0].axhline(np.mean(f1_micros), color="steelblue", linestyle="--", alpha=0.5)
axes[0].set_xlabel("Fold")
axes[0].set_ylabel("Score")
axes[0].set_title("F1 Scores Across Folds")
axes[0].set_ylim(0.74, 0.82)
axes[0].legend()
axes[0].set_xticks(fold_nums)

# Per-label F1 heatmap across folds
all_labels = list(folds[0]["per_label"].keys())
heatmap_data = []
for f in folds:
    heatmap_data.append([f["per_label"][l]["f1"] for l in all_labels])
heatmap = np.array(heatmap_data)

im = axes[1].imshow(heatmap.T, cmap="RdYlGn", aspect="auto", vmin=0.3, vmax=1.0)
axes[1].set_xticks(range(5))
axes[1].set_xticklabels([f"Fold {i+1}" for i in range(5)])
axes[1].set_yticks(range(len(all_labels)))
axes[1].set_yticklabels(all_labels)
axes[1].set_title("Per-Label F1 Heatmap Across Folds")

for i in range(len(all_labels)):
    for j in range(5):
        axes[1].text(j, i, f'{heatmap[j, i]:.2f}', ha='center', va='center', fontsize=8,
                    color='white' if heatmap[j, i] < 0.6 else 'black')

plt.colorbar(im, ax=axes[1], shrink=0.8)
plt.tight_layout()
plt.savefig("../notebooks/fig_fold_stability.png", dpi=150, bbox_inches='tight')
plt.show()

## 4. Precision vs Recall per Label

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

for l in labels:
    p = kfold_results["per_label_average"][l]["precision_mean"]
    r = kfold_results["per_label_average"][l]["recall_mean"]
    sup = kfold_results["per_label_average"][l]["avg_support"]
    ax.scatter(r, p, s=sup * 2, alpha=0.7, edgecolors='black', linewidth=0.5)
    ax.annotate(l, (r, p), textcoords="offset points", xytext=(8, 5), fontsize=10)

# F1 iso-lines
for f1 in [0.5, 0.6, 0.7, 0.8, 0.9]:
    r_range = np.linspace(0.01, 1, 100)
    p_range = f1 * r_range / (2 * r_range - f1)
    valid = (p_range > 0) & (p_range <= 1)
    ax.plot(r_range[valid], p_range[valid], '--', color='gray', alpha=0.3)
    # Label the iso-line
    idx = np.argmin(np.abs(r_range - 0.95))
    if valid[idx]:
        ax.text(0.96, p_range[idx], f'F1={f1}', fontsize=8, color='gray')

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision vs Recall per Label (bubble size = support)")
ax.set_xlim(0.35, 1.05)
ax.set_ylim(0.35, 1.05)
ax.set_aspect('equal')

plt.tight_layout()
plt.savefig("../notebooks/fig_precision_recall.png", dpi=150, bbox_inches='tight')
plt.show()